# 02 — Train YOLOv12

Trains a YOLOv12 model on the fire & smoke dataset.

| | |
|---|---|
| **Dataset** | `fire-smoke-new` — 115,845 train / 16,100 val |
| **Classes** | `0: smoke`, `1: fire` |
| **Model** | YOLOv12n (nano — fast, good for starting) |
| **Weights saved to** | `/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/runs/` |

---

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install ultralytics -q

import ultralytics
print('Ultralytics version:', ultralytics.__version__)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os
from ultralytics.utils import SETTINGS

DRIVE_TAR    = '/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/dataset/dataset.tar.gz'
LOCAL_DIR    = '/content/dataset'
DATASET_ROOT = '/content/dataset/fire-smoke-new'

if not os.path.exists(DATASET_ROOT):
    os.makedirs(LOCAL_DIR, exist_ok=True)
    print('Copying archive from Drive...')
    shutil.copy2(DRIVE_TAR, '/content/dataset_raw')
    print('Extracting...')
    !tar -xf /content/dataset_raw -C {LOCAL_DIR}
    print('Dataset ready!')
else:
    print('Dataset already exists, skipping copy.')

# Tell Ultralytics where datasets live
SETTINGS.update({'datasets_dir': '/content/dataset'})
print(f'\nDataset root  : {DATASET_ROOT}')
print(f'datasets_dir  : {SETTINGS["datasets_dir"]}')

In [ ]:
# Write data.yaml — tells YOLO where the data is and what the classes are
import yaml

DATA_YAML = '/content/data.yaml'

config = {
    'path' : DATASET_ROOT,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : 2,
    'names': {0: 'smoke', 1: 'fire'}
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('data.yaml written:')
print(open(DATA_YAML).read())

In [ ]:
from ultralytics import YOLO

DRIVE_RUNS = '/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/runs'

model = YOLO('yolo12n.pt')

results = model.train(
    data        = DATA_YAML,
    epochs      = 50,
    imgsz       = 640,
    batch       = 16,
    name        = 'fire_smoke_v1',
    project     = DRIVE_RUNS,
    patience    = 10,
    device      = 0,
    workers     = 4,
    cache       = True,
    save_period = 5,    # save checkpoint every 5 epochs (epoch5.pt, epoch10.pt, ...)
)

In [ ]:
# Show training curves
from IPython.display import Image
Image(f'{DRIVE_RUNS}/fire_smoke_v1/results.png')

In [ ]:
# Validate with best weights
best_weights = f'{DRIVE_RUNS}/fire_smoke_v1/weights/best.pt'
best = YOLO(best_weights)
metrics = best.val(data=DATA_YAML, device=0)

print('\n=== Validation Metrics ===')
print(f'  mAP@0.5      : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')

---
## ✅ Done!

Best weights saved to:
```
/content/drive/MyDrive/AI_TRAINING/PCCC_yolo/runs/fire_smoke_v1/weights/best.pt
```

➡️ Next: **03 — Inference & Testing**